# Last.fm Scraper — 6 Workers in 1 Window

**Run this notebook in a single VSCode window or Jupyter tab.**

- 6 parallel workers run inside one notebook using threads
- All workers share one done-set and write to one parquet
- Set your range in Cell 2 and run all cells

In [ ]:
import os, time, threading
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from filelock import FileLock

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SET THESE before running                                   ║
# ╚══════════════════════════════════════════════════════════════╝
TOTAL_WORKERS        = 6     # number of parallel workers
SAVE_EVERY           = 30    # flush to parquet every N rows per worker
SLEEP_BETWEEN_ALBUMS = 1.5   # seconds between each album per worker

# ╔══════════════════════════════════════════════════════════════╗
# ║  RANGE  (set both to None to scrape the full dataset)       ║
# ║  Example: START_FROM = 300000  END_AT = 600000              ║
# ╚══════════════════════════════════════════════════════════════╝
START_FROM = None   # e.g. 300000
END_AT     = None   # e.g. 600000

In [ ]:
# ── Paths (notebook lives in 1-data/ so data is one level up) ────────────────
PARQUET_PATH = '../data/mb_album_artists.parquet'
OUT_PATH     = '../data/lastfm_data.parquet'
LOCK_PATH    = '../data/lastfm_data.parquet.lock'

COLS = ['Artist', 'Album',
        'Artist_Listeners', 'Artist_Scrobbles',
        'Album_Listeners',  'Album_Scrobbles',
        'Similar_Artists',  'Artist_URL', 'Album_URL']

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

lock       = FileLock(LOCK_PATH, timeout=60)
print_lock = threading.Lock()

_total_saved   = 0
_total_skipped = 0
_counter_lock  = threading.Lock()

def log(worker_id, msg):
    with print_lock:
        print(f'[W{worker_id}] {msg}', flush=True)

expected = round(60 / (SLEEP_BETWEEN_ALBUMS + 2) * TOTAL_WORKERS)
print(f'Workers              : {TOTAL_WORKERS}')
print(f'Sleep between albums : {SLEEP_BETWEEN_ALBUMS}s')
print(f'Expected speed       : ~{expected} albums/min')

## (Optional) Import existing CSV
If you already have scraped data in a CSV, run this cell once to import it into the shared parquet.  
**Skip this cell if you have no CSV** — it does nothing if `CSV_IMPORT_PATH` does not exist.

In [ ]:
CSV_IMPORT_PATH = 'data/lastfm_data.csv'

if not os.path.exists(CSV_IMPORT_PATH):
    print('No CSV found — skipping import.')
else:
    df_csv = pd.read_csv(CSV_IMPORT_PATH)
    df_csv.columns = [c.strip().replace(' ', '_') for c in df_csv.columns]
    for col in COLS:
        if col not in df_csv.columns:
            df_csv[col] = 'N/A'
    df_csv = df_csv[COLS].copy()
    print(f'CSV rows loaded: {len(df_csv):,}')
    with lock:
        if os.path.exists(OUT_PATH):
            df_existing = pd.read_parquet(OUT_PATH)
            df_out = pd.concat([df_existing, df_csv], ignore_index=True)
        else:
            df_out = df_csv
        before = len(df_out)
        df_out = df_out.drop_duplicates(subset=['Artist', 'Album']).reset_index(drop=True)
        df_out.to_parquet(OUT_PATH, index=False)
    print(f'Imported {len(df_out):,} rows  ({before - len(df_out):,} duplicates dropped)')

In [ ]:
# ── Load source & compute range ──────────────────────────────────────────────
df_all = pd.read_parquet(PARQUET_PATH)
artist_col = 'artist_name' if 'artist_name' in df_all.columns else 'name'
album_col  = 'album_name'  if 'album_name'  in df_all.columns else 'album'

df_unique = (
    df_all[[artist_col, album_col]]
    .drop_duplicates()
    .reset_index(drop=True)
)
total = len(df_unique)

start = max(0, START_FROM) if START_FROM is not None else 0
end   = min(END_AT, total) if END_AT   is not None else total
df_range = df_unique.iloc[start:end].reset_index(drop=True)

# Split df_range evenly across workers
chunk    = (len(df_range) + TOTAL_WORKERS - 1) // TOTAL_WORKERS
slices   = [df_range.iloc[i*chunk : min((i+1)*chunk, len(df_range))].reset_index(drop=True)
            for i in range(TOTAL_WORKERS) if i*chunk < len(df_range)]

print(f'Total unique albums  : {total:,}')
print(f'Range                : rows {start:,} to {end:,}  ({len(df_range):,} albums)')
print(f'Workers              : {len(slices)}')
for i, s in enumerate(slices):
    print(f'  Worker {i}: {len(s):,} albums')

In [ ]:
# ── Load done-set (shared across all workers) ────────────────────────────────
def load_done_set():
    if not os.path.exists(OUT_PATH):
        return set()
    df_done = pd.read_parquet(OUT_PATH, columns=['Artist', 'Album'])
    return set(zip(df_done['Artist'].str.lower(), df_done['Album'].str.lower()))

done_set      = load_done_set()
done_set_lock = threading.Lock()

print(f'Already scraped: {len(done_set):,}')

In [ ]:
# ── Scraper & flush helpers ──────────────────────────────────────────────────

def scrape_artist(url):
    listeners, scrobbles, similar = 'N/A', 'N/A', 'None Found'
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code == 200:
                soup = BeautifulSoup(r.text, 'html.parser')
                container = soup.find('div', class_='header-new-info-desktop')
                if container:
                    for item in container.find_all('li', class_='header-metadata-tnew-item'):
                        title = item.find('h4', class_='header-metadata-tnew-title')
                        abbr  = item.find('abbr', class_='js-abbreviated-counter')
                        if title and abbr:
                            if 'Listeners'  in title.text: listeners = abbr.get('title')
                            elif 'Scrobbles' in title.text: scrobbles = abbr.get('title')
                sims = [a.text.strip()
                        for h in soup.find_all('h3', class_='catalogue-overview-similar-artists-item-name')
                        for a in [h.find('a')] if a]
                if sims: similar = ', '.join(sims)
                if listeners != 'N/A': break
            time.sleep(2)
        except Exception as e:
            time.sleep(2)
    return listeners, scrobbles, similar

def scrape_album(url):
    listeners, scrobbles = 'N/A', 'N/A'
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code == 200:
                soup  = BeautifulSoup(r.text, 'html.parser')
                abbrs = soup.find_all('abbr', class_='js-abbreviated-counter')
                if len(abbrs) >= 2:
                    listeners = abbrs[0].get('title', 'N/A')
                    scrobbles = abbrs[1].get('title', 'N/A')
                elif len(abbrs) == 1:
                    listeners = abbrs[0].get('title', 'N/A')
                if listeners != 'N/A' and scrobbles != 'N/A': break
            time.sleep(2)
        except Exception as e:
            time.sleep(2)
    return listeners, scrobbles

def flush_buffer(buffer):
    global _total_saved
    df_new = pd.DataFrame(buffer, columns=COLS)
    with lock:
        if os.path.exists(OUT_PATH):
            df_out = pd.concat([pd.read_parquet(OUT_PATH), df_new], ignore_index=True)
        else:
            df_out = df_new
        df_out = df_out.drop_duplicates(subset=['Artist', 'Album']).reset_index(drop=True)
        df_out.to_parquet(OUT_PATH, index=False)
        n = len(df_out)
    with _counter_lock:
        _total_saved = n
    return n

print('Helpers ready.')

In [ ]:
# ── Worker function ──────────────────────────────────────────────────────────
def worker(worker_id, df_slice):
    global _total_skipped
    buffer  = []
    scraped = 0
    skipped = 0
    total   = len(df_slice)

    for i, row in df_slice.iterrows():
        artist = str(row[artist_col]).strip()
        album  = str(row[album_col]).strip()

        if not artist or artist.lower() in ('none', 'nan'):
            continue

        key = (artist.lower(), album.lower())

        # ── Check before scraping ────────────────────────────────────────────
        with done_set_lock:
            if key in done_set:
                skipped += 1
                with _counter_lock:
                    _total_skipped += 1
                log(worker_id, f'SKIP [{i+1}/{total}] {artist} — {album}')
                continue

        # ── Scrape ───────────────────────────────────────────────────────────
        a_slug     = artist.replace(' ', '+')
        al_slug    = album.replace(' ', '+')
        artist_url = f'https://www.last.fm/music/{a_slug}'
        album_url  = f'https://www.last.fm/music/{a_slug}/{al_slug}'

        log(worker_id, f'[{i+1}/{total}] {artist} — {album}')

        a_listeners, a_scrobbles, similar = scrape_artist(artist_url)
        time.sleep(1)
        al_listeners, al_scrobbles = scrape_album(album_url)

        buffer.append([artist, album,
                       a_listeners, a_scrobbles,
                       al_listeners, al_scrobbles,
                       similar, artist_url, album_url])

        with done_set_lock:
            done_set.add(key)
        scraped += 1

        # ── Flush every SAVE_EVERY rows ──────────────────────────────────────
        if len(buffer) >= SAVE_EVERY:
            n = flush_buffer(buffer)
            buffer = []
            log(worker_id, f'-- saved -- total in parquet: {n:,}')

        time.sleep(SLEEP_BETWEEN_ALBUMS)   # ← controlled by setting at top

    # Final flush
    if buffer:
        n = flush_buffer(buffer)
        buffer = []
        log(worker_id, f'-- final flush -- total in parquet: {n:,}')

    log(worker_id, f'DONE — scraped: {scraped:,}  skipped: {skipped:,}')

# ── Launch all workers ───────────────────────────────────────────────────────
import time as _time
print(f'Launching {len(slices)} workers...')
print('-' * 60)

threads    = []
start_time = _time.time()

for wid, df_slice in enumerate(slices):
    t = threading.Thread(target=worker, args=(wid, df_slice), daemon=True)
    threads.append(t)
    t.start()

try:
    for t in threads:
        t.join()
except KeyboardInterrupt:
    print('\nStopped — progress is saved in parquet.')

elapsed = _time.time() - start_time
print('\n' + '=' * 60)
print(f'All workers finished in {elapsed/60:.1f} minutes')
print(f'Total saved in parquet : {_total_saved:,}')
print(f'Total skipped          : {_total_skipped:,}')

---
## Status — run from any window at any time

In [ ]:
if os.path.exists(OUT_PATH):
    df_status = pd.read_parquet(OUT_PATH)
    print(f'Total rows in parquet : {len(df_status):,}')
    print(f'Unique artists        : {df_status["Artist"].nunique():,}')
    print(f'File size             : {os.path.getsize(OUT_PATH)/1024/1024:.1f} MB')
    display(df_status.tail(5))
else:
    print('Parquet not created yet — run the scraper first.')